# One type, three semantics

> When is consensus among independent Cogs required?
>
> — [@oliphant2026, §5.6]

The manifest answers with a threshold: `consensus_disagreement > 0.25`.
But the rule does not say what the number measures. Rather than choose
for the author, the specification fixes the metric's **type** (a
fact-by-reader answer matrix, answers in {agree, no, cantTell}, mapped
to a scalar in [0, 1] against the factored threshold) and exhibits
three type-satisfying metrics with different semantics, evaluated on
the same matrix. All three are correctly constructed. They do not agree
on whether the gate fires.

**A correctly constructed policy is not the same as a fit-for-purpose
policy.** What is right for the job here is an open algorithmic policy
design question, likely to have no unique correct answer but many
viable answers with different implications on incentives, and the right
answer is not limited to these three.

## The type, in the model

The abstract definition carries the type signature; three concrete
definitions specialize it, each documenting its semantics and its
incentive implication. The comparator oracle's metric slot is typed by
the abstract definition and deliberately left unbound: the model states
that a metric of this type is required and refuses to choose one
(GAP-04 in the appendix, adjudicated open).

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.show_metric_definitions()

abstract part def DisagreementMetric {
            doc /* adjudicated: GAP-04 — the rule does not say what the
               number is, and no metric is chosen HERE. This abstract def
               is the type signature: a metric maps a fact-by-reader
               answer matrix (answers in {agree, no, cantTell}) to a
               scalar in [0, 1] comparable against the manifest's 0.25
               threshold. Three type-satisfying alternatives with different
               semantics follow; which is fit for purpose is an open
               algorithmic policy design question, unlikely to have a
               unique correct answer, with different implications on
               incentives. The right answer is not limited to these three. */
            attribute metricName : ScalarValues::String;
        }

part def DissentFractionMetric :> DisagreementMetric {
            doc /* Fraction of reader answers differing from each fact's
               modal expressed value, counting c

## The three implementations

The same three semantics as running code, satisfying the declared type:

- **dissent-fraction** counts every answer off the modal value,
  cantTell included: honest abstention is punished, so readers are
  pushed to guess. On the paper's own section 5.5 example (three Cogs,
  two must agree) it FIRES the gate at 0.25, contradicting the
  example's intent.
- **strict-quorum-undecodable** counts facts where no value reaches a
  quorum of two expressed answers; abstention blocks quorum, so it
  escalates more, spending expert-review capacity.
- **erasure-aware-undecodable** treats cantTell as an erasure, not an
  error: a lone expressed voice among abstentions decodes, so
  abstention shifts power to whoever still answers.

In [2]:
import inspect
for f in (exhibits.dissent_fraction, exhibits.strict_quorum_undecodable,
          exhibits.erasure_aware_undecodable):
    print(inspect.getsource(f))

def dissent_fraction(matrix):
    dissents = total = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        modal = max(set(expressed), key=expressed.count) if expressed else None
        dissents += sum(1 for v in fact if v != modal)
        total += len(fact)
    return dissents / total

def strict_quorum_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        if not expressed or max(expressed.count(v) for v in set(expressed)) < 2:
            undecodable += 1
    return undecodable / len(matrix)

def erasure_aware_undecodable(matrix):
    undecodable = 0
    for fact in matrix:
        expressed = [v for v in fact if v != "?"]
        if not expressed or max(expressed.count(v) for v in set(expressed)) * 2 <= len(expressed):
            undecodable += 1
    return undecodable / len(matrix)



## One matrix, three answers

The same answer matrix, the threshold read from the model's single
point of definition, and three different gate decisions. The Track
records which metric a run actually used; recording is not
endorsing.

In [3]:
exhibits.evaluate_metrics()

type signature: DisagreementMetric = (facts x readers answer matrix, '?' = cantTell) -> [0, 1]

same answer matrix for all three metrics (8 facts x 3 readers):
  f1: A A A
  f2: A A A
  f3: A A A
  f4: A A B
  f5: A A ?
  f6: A B ?
  f7: A B C
  f8: A ? ?
threshold (from the model, single point of definition): > 0.25

dissent-fraction          : 0.333 -> gate fires: True
strict-quorum-undecodable : 0.375 -> gate fires: True
erasure-aware-undecodable : 0.250 -> gate fires: False
